In [1]:
import os
import wave

import torch
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from utils8 import AudioCNN
from utils8.data import AudioDataset, get_dataloader

# Get Data
path = os.path.join('Data', 'Digits')
classes = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
dataset = AudioDataset(path, classes)

# Data Split and Subsets
idx = list(range(len(dataset)))
labels = dataset.labels
train_val_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

In [30]:
from IPython.display import Audio, display

N_SAMPLES = 5
sample_audio = [train_val_set[i][0] for i in range(N_SAMPLES)]
y = [train_val_set[i][1] for i in range(N_SAMPLES)]

for audio in sample_audio:
    display(Audio(audio, rate=sr))

# Different Data Augmentations

## Backgroudn Noise

In [31]:
from torchaudio import functional as F
import torch


def add_gaussian_nosie(wave:torch.Tensor, snr_db:torch.Tensor = torch.tensor([20, 10, 3])) -> torch.Tensor:
    noise = torch.randn_like(wave)
    return F.add_noise(wave, noise, snr_db)

for audio in sample_audio:
    new_audio = add_gaussian_nosie(audio)
    display(Audio(new_audio, rate=sr))

## Pitch Shift

In [33]:
from torchaudio import functional as F
import torchaudio.transforms as T
import torch


def pitch_shift(waveform: torch.Tensor, n_steps: float = 8, sr: int=8000, n_fft: int = 512, hop_length: int = 160, win_length: int = 400,) -> torch.Tensor:
    rate = 2 ** (-n_steps / 12)           # <1 = skróć (wyższy ton), >1 = wydłuż

    # Spektrogram zespolony (power=None)
    spec_fn = T.Spectrogram(
        n_fft=n_fft,
        win_length=win_length,
        hop_length=hop_length,
        power=None,          # zwraca complex tensor
    )
    spec = spec_fn(waveform)               # (1, F, T_frames) complex

    # Time-stretch przez Phase Vocoder
    stretch_fn = T.TimeStretch(
        hop_length=hop_length,
        n_freq=n_fft // 2 + 1,
        fixed_rate=rate,
    )
    stretched_spec = stretch_fn(spec)     # (1, F, T_new) complex

    # Rekonstrukcja → (1, T_stretched)
    inv_spec_fn = T.InverseSpectrogram(
        n_fft=n_fft,
        win_length=win_length,
        hop_length=hop_length,
    )
    stretched_wav = inv_spec_fn(stretched_spec)

    # Resample do oryginalnej długości (to właśnie przesuwa ton)
    orig_len     = waveform.shape[-1]
    stretched_len = stretched_wav.shape[-1]
    resampled = F.resample(stretched_wav, orig_freq=stretched_len, new_freq=orig_len)

    # Przytnij / dopełnij do dokładnej długości
    if resampled.shape[-1] > orig_len:
        resampled = resampled[..., :orig_len]
    elif resampled.shape[-1] < orig_len:
        pad = orig_len - resampled.shape[-1]
        resampled = torch.nn.functional.pad(resampled, (0, pad))

    return resampled

for audio in sample_audio:
    new_audio = pitch_shift(audio)
    display(Audio(new_audio, rate=sr))

## Reverb


In [36]:
def add_reverb(waveform: torch.Tensor, sr: int = 8000, room_scale: float = 0.4) -> torch.Tensor:
    ir_len = int(sr * room_scale * 0.5)
    t_ir   = torch.linspace(0, 1, ir_len)
    decay  = torch.exp(-6 * t_ir)
    ir     = torch.randn(ir_len) * decay
    ir     = ir / ir.abs().max().clamp(min=1e-8)       # (ir_len,)

    # conv1d wymaga (batch, channels, length)
    wav_3d = waveform.unsqueeze(0)                     # (1, 1, T)
    ir_3d  = ir.flip(0).unsqueeze(0).unsqueeze(0)      # (1, 1, ir_len) – kernel
    padding = ir_len - 1

    reverbed = torch.nn.functional.conv1d(
        wav_3d, ir_3d, padding=padding
    )[..., :waveform.shape[-1]]

    reverbed = reverbed.squeeze(0)                     # → (1, T)

    # Normalizacja amplitudy
    scale = waveform.abs().max() / reverbed.abs().max().clamp(min=1e-8)
    return reverbed * scale

for audio in sample_audio:
    new_audio = add_reverb(audio)
    display(Audio(new_audio, rate=sr))

# Equalization

In [41]:
def bass_boost(waveform: torch.Tensor, sr: int = 8000, gain_db: float = 12.0) -> torch.Tensor:
    filtered = F.lowpass_biquad(waveform, sample_rate=sr, cutoff_freq=300.0)
    gain_lin  = 10 ** (gain_db / 20)
    bass_only = filtered * gain_lin

    # Blend
    return waveform + (bass_only - waveform) * 0.6

for audio in sample_audio:
    new_audio = bass_boost(audio)
    display(Audio(new_audio, rate=sr))